In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("merged_and_cleaned_all_data.parquet", engine="pyarrow")

In [3]:
df.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1
2,Potato and Fennel Soup Hodge,In a heavy saucepan cook diced fennel and oni...,"[{""name"": ""fennel bulb (sometimes called anise...",40.0,165.0,7.0,6.0,garlish,NaN,2
3,Mahi-Mahi in Tomato Olive Sauce,Heat oil in heavy skillet over -high heat. Ad...,"[{""name"": ""extra-virgin olive oil"", ""quantity""...",22.0,NaN,NaN,NaN,main dish,NaN,3
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4


In [4]:
pd.set_option('display.max_colwidth', None)
print(df.iloc[144831].Ingredients)

[{"name": "glazed doughnut", "quantity": 6.0, "unit": "piece"}, {"name": "butter", "quantity": 0.5, "unit": "cup"}, {"name": "cream cheese", "quantity": 8.0, "unit": "ounce"}, {"name": "powdered sugar", "quantity": 1.5, "unit": "cup"}, {"name": "pure vanilla extract", "quantity": 1.0, "unit": "teaspoon"}, {"name": "Cool Whip", "quantity": 1.5, "unit": "cup"}, {"name": "marshmallow fluff", "quantity": 1.0, "unit": "cup"}, {"name": "variou fruit for topping. just make sure you slice enough. if you use banana i recommend coating them with lemon juice to avoid browning. pick your favorite fruit combination!!", "quantity": 1.0, "unit": "piece"}]


In [5]:
import pandas as pd
import numpy as np
import ast
import json

def extract_ingredient_names(ingredient_data):
    # 1) If NumPy array or Pandas Series
    if isinstance(ingredient_data, (np.ndarray, pd.Series)):
        if ingredient_data.size == 0 or pd.isna(ingredient_data).all():
            return []
        ingredient_data = ingredient_data.tolist()

    # 2) None or scalar NaN
    if ingredient_data is None:
        return []
    if isinstance(ingredient_data, float) and np.isnan(ingredient_data):
        return []

    # 3) Is Python List?
    if isinstance(ingredient_data, list):
        parsed_data = ingredient_data

    # 4) If it is Dict, it can be singular object, add to List
    elif isinstance(ingredient_data, dict):
        parsed_data = [ingredient_data]

    # 5) If it is String, first literal then JSON parse
    elif isinstance(ingredient_data, str):
        # First try literal_eval
        try:
            parsed_data = ast.literal_eval(ingredient_data)
            if isinstance(parsed_data, dict):
                parsed_data = [parsed_data]
            elif not isinstance(parsed_data, list):
                return []
        except (SyntaxError, ValueError):
            # literal_eval did not work, try JSON parse
            try:
                parsed_data = json.loads(ingredient_data)
                if isinstance(parsed_data, dict):
                    parsed_data = [parsed_data]
                elif not isinstance(parsed_data, list):
                    return []
            except (json.JSONDecodeError, TypeError):
                return []
    else:
        return []

    # 6) Checking parsed data is List?
    if not isinstance(parsed_data, list):
        return []

    # 7) If "quantity" and "unit" fields are none, skip the entity
    names = []
    for item in parsed_data:
        if isinstance(item, dict):
            quantity = item.get('quantity')
            unit = item.get('unit')
            # If "quantity" and "unit" fields are none, skip the entity
            if (quantity is None or quantity == "null") and (unit is None or unit == "null"):
                continue
            # If not, take "name" field
            name = item.get('name')
            if name:
                names.append(str(name))

    # 8) Return the List of ingredient names
    return names

In [6]:
ingredients = pd.DataFrame()
ingredients["Ingredients"] = df['Ingredients'].apply(extract_ingredient_names)

In [7]:
ingredients["ID"] = df["ID"]

In [8]:
import re
import string

import re
import string

def on_temizleme(strings):
    """
    Transformation rules:

    1) Remove everything inside parentheses.
    2) Reduce multiple spaces to a single space.
    3) Remove trailing punctuation marks.
    4) Remove leading spaces and punctuation marks.

    4.1) If the character before or after "/" is a number 
         (e.g., "10/20", "10/", "/20"), remove the numbers and the '/' character completely.

    4.5) If '/', '/possibly', or '/ possibly' appears, replace it with ' or '.

    4.6) If "kg", "ounce", "inch", or "oz" is not adjacent to a letter, 
         remove these words along with any attached non-whitespace characters.

    4.7) If the word "or" appears at the beginning of a sentence 
         (including punctuation, etc.), remove it as well.

    4.8) If the character "x" is not adjacent to a letter, remove the "x" character.

    4.9) Remove words containing numbers completely.

    5) Replace any characters that are not allowed (letters, numbers, '-', ''', ',') with a space.
    6) Again, reduce multiple spaces to a single space and trim leading/trailing spaces.
    """


    allowed_chars = set(string.ascii_letters + string.digits + "-',")

    cleaned_list = []
    for s in strings:
        # -------------------- 1) Delete inner field of Paranthesis --------------------
        s = re.sub(r"\(.*?\)", "", s)

        # -------------------- 2) Delete more then one spaces -----------------
        s = re.sub(r"\s+", " ", s).strip()

        # -------------------- 3) Delete punctuation at the end -------
        while s and s[-1] in string.punctuation:
            s = s[:-1]

        # -------------------- 4) Delete space/punctuation at the beginning --------
        while s and (s[0].isspace() or s[0] in string.punctuation):
            s = s[1:]

        # -------------------- 4.1) Delete slash + number ------------------
        s = re.sub(r"\d+/\d+", "", s)  # Örn. "10/20"
        s = re.sub(r"\d+/", "", s)     # Örn. "10/"
        s = re.sub(r"/\d+", "", s)     # Örn. "/20"

        # -------------------- 4.5) /possibly => ' or ' -----------------
        s = re.sub(r'/(\s*possibly)?', ' or ', s)

        # -------------------- 4.6) Delete kg, ounce, inch, oz -----------
        # If there is no letter before or after, remove the word along with any adjacent non-whitespace characters
        pattern_units = r'(?i)(?:(?<![a-zA-Z])[^a-zA-Z\s]*(?:kg|ounce|inch|oz|cm|mi)|(?:kg|ounce|inch|oz|cm|mi)[^a-zA-Z\s]*(?![a-zA-Z]))'
        s = re.sub(pattern_units, '', s)

        # -------------------- 4.7) Delete "or" at the beginning -------------
        s = re.sub(r'^[^a-zA-Z0-9]*or\b', '', s, flags=re.IGNORECASE)

        # -------------------- 4.8) Delete "x" if there is no char around of it ---------
        s = re.sub(r'(?i)(?<![a-z])x|x(?![a-z])', '', s)

        # -------------------- 4.9) Delete words which contain numbers ----------
        # \b -> word boundary, \w* -> 0 or more word characters,
        # \d+ -> at least one digit, followed by \w* again, \b
        s = re.sub(r"\b\w*\d+\w*\b", "", s)

        # -------------------- 5) Not allowed chars swapping with space ----
        temp = []
        for ch in s:
            if ch in allowed_chars or ch.isspace():
                temp.append(ch)
            else:
                temp.append(" ")
        new_s = "".join(temp)

        # -------------------- 6) Delete more than one spaces -----------
        new_s = re.sub(r"\s+", " ", new_s).strip()

        cleaned_list.append(new_s)

    return cleaned_list


In [9]:
ingredients["cleaned_ingredients"] = ingredients["Ingredients"].apply(on_temizleme)

In [10]:
ingredients.head()

,Ingredients,ID,cleaned_ingredients
0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas, turkey breast, thinly sliced, Bibb lettuce]",0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, whole-wheat lavash, cut in half crosswise, or flour tortillas, turkey breast, thinly sliced, Bibb lettuce]"
1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into 1-inch chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, minced, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or 3/4 teaspoon dried, crumbled, sugar]",1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, shallots, nced, butter, trimmed boneless center pork loin, sinew removed cut into chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, nced, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or teaspoon dried, crumbled, sugar]"
2,"[fennel bulb (sometimes called anise), stalks discarded, bulb cut into 1/2-inch dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet (baking) potatoes, chicken broth, milk]",2,"[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet potatoes, chicken broth, lk]"
3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, 14 1/2-ounce cans diced tomatoes with garlic, basil, and oregano in juice, 6-ounce mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, (packed) finely grated orange peel]",3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, cans diced tomatoes with garlic, basil, and oregano in juice, mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, finely grated orange peel]"
4,"[12-ounce package frozen spinach soufflé, thawed, extra-wide egg noodles, freshly cooked, sour cream, purchased pesto sauce, ground nutmeg, grated sharp cheddar cheese]",4,"[package frozen spinach souffle , thawed, extra-wide egg noodles, freshly cooked, sour cream, purchased pesto sauce, ground nutmeg, grated sharp cheddar cheese]"


In [11]:
def lists_to_dict_for_row(row, key_col="Ingredients", value_col="cleaned_ingredients"):
    keys = row[key_col]
    values = row[value_col]

    if not isinstance(keys, list) or not isinstance(values, list):
        return {}

    min_len = min(len(keys), len(values))
    merged_dict = dict(zip(keys[:min_len], values[:min_len]))

    return merged_dict

In [14]:
first_map = pd.DataFrame()
first_map["ID"] = ingredients["ID"]
first_map["Dict"] = ingredients.apply(lists_to_dict_for_row, axis=1)


In [15]:
first_map.head()

,ID,Dict
0,0,"{'low-sodium vegetable or chicken stock': 'low-sodium vegetable or chicken stock', 'dried brown lentils': 'dried brown lentils', 'dried French green lentils': 'dried French green lentils', 'celery, chopped': 'celery, chopped', 'carrot, peeled and chopped': 'carrot, peeled and chopped', 'fresh thyme': 'fresh thyme', 'kosher salt': 'kosher salt', 'tomato, cored, seeded, and diced': 'tomato, cored, seeded, and diced', 'Fuji apple, cored and diced': 'Fuji apple, cored and diced', 'freshly squeezed lemon juice': 'freshly squeezed lemon juice', 'extra-virgin olive oil': 'extra-virgin olive oil', 'whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas': 'whole-wheat lavash, cut in half crosswise, or flour tortillas', 'turkey breast, thinly sliced': 'turkey breast, thinly sliced', 'Bibb lettuce': 'Bibb lettuce'}"
1,1,"{'whipping cream': 'whipping cream', 'onions, chopped': 'onions, chopped', 'salt': 'salt', 'bay leaves': 'bay leaves', 'whole cloves': 'whole cloves', 'garlic clove, crushed': 'garlic clove, crushed', 'pepper': 'pepper', 'ground nutmeg': 'ground nutmeg', 'shallots, minced': 'shallots, nced', 'butter': 'butter', 'trimmed boneless center pork loin, sinew removed cut into 1-inch chunks, well chilled': 'trimmed boneless center pork loin, sinew removed cut into chunks, well chilled', 'eggs': 'eggs', 'all purpose flour': 'all purpose flour', 'tawny Port': 'tawny Port', 'dried currants, minced': 'dried currants, nced', 'olive oil': 'olive oil', 'red onions, halved, sliced': 'red onions, halved, sliced', 'dried currants': 'dried currants', 'red wine vinegar': 'red wine vinegar', 'canned chicken broth': 'canned chicken broth', 'chopped fresh thyme or 3/4 teaspoon dried, crumbled': 'chopped fresh thyme or teaspoon dried, crumbled', 'sugar': 'sugar'}"
2,2,"{'fennel bulb (sometimes called anise), stalks discarded, bulb cut into 1/2-inch dice, and feathery leaves reserved for garnish': 'fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish', 'onion, diced': 'onion, diced', 'unsalted butter': 'unsalted butter', 'russet (baking) potatoes': 'russet potatoes', 'chicken broth': 'chicken broth', 'milk': 'lk'}"
3,3,"{'extra-virgin olive oil': 'extra-virgin olive oil', 'chopped onion': 'chopped onion', 'dry white wine': 'dry white wine', 'anchovy paste': 'anchovy paste', '14 1/2-ounce cans diced tomatoes with garlic, basil, and oregano in juice': 'cans diced tomatoes with garlic, basil, and oregano in juice', '6-ounce mahi-mahi fillets': 'mahi-mahi fillets', 'green olives, quartered, pitted': 'green olives, quartered, pitted', 'chopped fresh oregano, divided': 'chopped fresh oregano, divided', '(packed) finely grated orange peel': 'finely grated orange peel'}"
4,4,"{'12-ounce package frozen spinach soufflé, thawed': 'package frozen spinach souffle , thawed', 'extra-wide egg noodles, freshly cooked': 'extra-wide egg noodles, freshly cooked', 'sour cream': 'sour cream', 'purchased pesto sauce': 'purchased pesto sauce', 'ground nutmeg': 'ground nutmeg', 'grated sharp cheddar cheese': 'grated sharp cheddar cheese'}"


In [16]:
first_map.to_csv("first_map.csv")

In [17]:
def process_list(ingredient_list):
    """Her satırdaki listeyi işler ve iç içe listeler halinde temizlenmiş kelimeler döndürür."""
    if not isinstance(ingredient_list, list):
        return []  

    processed_words = []
    for ingredient in ingredient_list:
        if not isinstance(ingredient, str):
            continue  

        split_ingredients = ingredient.split(" or ")

        cleaned_sublist = []
        for item in split_ingredients:
            item = re.sub(r"[^,\w\s]", " ", item)
            item = re.sub(r"\s+", " ", item).strip()

            if item:  
                cleaned_sublist.append(item)

        processed_words.append(cleaned_sublist)

    return processed_words

In [18]:
ingredients["splitted_ingredients"] = ingredients["cleaned_ingredients"].apply(process_list)

In [19]:
ingredients.head()

,Ingredients,ID,cleaned_ingredients,splitted_ingredients
0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas, turkey breast, thinly sliced, Bibb lettuce]",0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, whole-wheat lavash, cut in half crosswise, or flour tortillas, turkey breast, thinly sliced, Bibb lettuce]","[[low sodium vegetable, chicken stock], [dried brown lentils], [dried French green lentils], [celery, chopped], [carrot, peeled and chopped], [fresh thyme], [kosher salt], [tomato, cored, seeded, and diced], [Fuji apple, cored and diced], [freshly squeezed lemon juice], [extra virgin olive oil], [whole wheat lavash, cut in half crosswise,, flour tortillas], [turkey breast, thinly sliced], [Bibb lettuce]]"
1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into 1-inch chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, minced, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or 3/4 teaspoon dried, crumbled, sugar]",1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, shallots, nced, butter, trimmed boneless center pork loin, sinew removed cut into chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, nced, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or teaspoon dried, crumbled, sugar]","[[whipping cream], [onions, chopped], [salt], [bay leaves], [whole cloves], [garlic clove, crushed], [pepper], [ground nutmeg], [shallots, nced], [butter], [trimmed boneless center pork loin, sinew removed cut into chunks, well chilled], [eggs], [all purpose flour], [tawny Port], [dried currants, nced], [olive oil], [red onions, halved, sliced], [dried currants], [red wine vinegar], [canned chicken broth], [chopped fresh thyme, teaspoon dried, crumbled], [sugar]]"
2,"[fennel bulb (sometimes called anise), stalks discarded, bulb cut into 1/2-inch dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet (baking) potatoes, chicken broth, milk]",2,"[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet potatoes, chicken broth, lk]","[[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish], [onion, diced], [unsalted butter], [russet potatoes], [chicken broth], [lk]]"
3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, 14 1/2-ounce cans diced tomatoes with garlic, basil, and oregano in juice, 6-ounce mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, (packed) finely grated orange peel]",3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, cans diced tomatoes with garlic, basil, and oregano in juice, mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, finely grated orange peel]","[[extra virgin olive oil], [chopped onion], [dry white wine], [anchovy paste], [cans diced tomatoes with garlic, basil, and oregano in juice], [mahi mahi fillets], [green olives, quartered, pitted], [chopped fresh oregano, divided], [finely grated orange peel]]"
4,"[12-ounce package frozen spinach soufflé, thawed, extr

In [20]:
def lists_to_dict_for_row(row, key_col="cleaned_ingredients", value_col="splitted_ingredients"):
    keys = row[key_col]
    values = row[value_col]

    if not isinstance(keys, list) or not isinstance(values, list):
        return {}

    min_len = min(len(keys), len(values))
    merged_dict = dict(zip(keys[:min_len], values[:min_len]))

    return merged_dict

In [21]:
second_map = pd.DataFrame()
second_map["ID"] = ingredients["ID"]
second_map["Dict"] = ingredients.apply(lists_to_dict_for_row, axis=1)

In [22]:
second_map.head()

,ID,Dict
0,0,"{'low-sodium vegetable or chicken stock': ['low sodium vegetable', 'chicken stock'], 'dried brown lentils': ['dried brown lentils'], 'dried French green lentils': ['dried French green lentils'], 'celery, chopped': ['celery, chopped'], 'carrot, peeled and chopped': ['carrot, peeled and chopped'], 'fresh thyme': ['fresh thyme'], 'kosher salt': ['kosher salt'], 'tomato, cored, seeded, and diced': ['tomato, cored, seeded, and diced'], 'Fuji apple, cored and diced': ['Fuji apple, cored and diced'], 'freshly squeezed lemon juice': ['freshly squeezed lemon juice'], 'extra-virgin olive oil': ['extra virgin olive oil'], 'whole-wheat lavash, cut in half crosswise, or flour tortillas': ['whole wheat lavash, cut in half crosswise,', 'flour tortillas'], 'turkey breast, thinly sliced': ['turkey breast, thinly sliced'], 'Bibb lettuce': ['Bibb lettuce']}"
1,1,"{'whipping cream': ['whipping cream'], 'onions, chopped': ['onions, chopped'], 'salt': ['salt'], 'bay leaves': ['bay leaves'], 'whole cloves': ['whole cloves'], 'garlic clove, crushed': ['garlic clove, crushed'], 'pepper': ['pepper'], 'ground nutmeg': ['ground nutmeg'], 'shallots, nced': ['shallots, nced'], 'butter': ['butter'], 'trimmed boneless center pork loin, sinew removed cut into chunks, well chilled': ['trimmed boneless center pork loin, sinew removed cut into chunks, well chilled'], 'eggs': ['eggs'], 'all purpose flour': ['all purpose flour'], 'tawny Port': ['tawny Port'], 'dried currants, nced': ['dried currants, nced'], 'olive oil': ['olive oil'], 'red onions, halved, sliced': ['red onions, halved, sliced'], 'dried currants': ['dried currants'], 'red wine vinegar': ['red wine vinegar'], 'canned chicken broth': ['canned chicken broth'], 'chopped fresh thyme or teaspoon dried, crumbled': ['chopped fresh thyme', 'teaspoon dried, crumbled'], 'sugar': ['sugar']}"
2,2,"{'fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish': ['fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish'], 'onion, diced': ['onion, diced'], 'unsalted butter': ['unsalted butter'], 'russet potatoes': ['russet potatoes'], 'chicken broth': ['chicken broth'], 'lk': ['lk']}"
3,3,"{'extra-virgin olive oil': ['extra virgin olive oil'], 'chopped onion': ['chopped onion'], 'dry white wine': ['dry white wine'], 'anchovy paste': ['anchovy paste'], 'cans diced tomatoes with garlic, basil, and oregano in juice': ['cans diced tomatoes with garlic, basil, and oregano in juice'], 'mahi-mahi fillets': ['mahi mahi fillets'], 'green olives, quartered, pitted': ['green olives, quartered, pitted'], 'chopped fresh oregano, divided': ['chopped fresh oregano, divided'], 'finely grated orange peel': ['finely grated orange peel']}"
4,4,"{'package frozen spinach souffle , thawed': ['package frozen spinach souffle , thawed'], 'extra-wide egg noodles, freshly cooked': ['extra wide egg noodles, freshly cooked'], 'sour cream': ['sour cream'], 'purchased pesto sauce': ['purchased pesto sauce'], 'ground nutmeg': ['ground nutmeg'], 'grated sharp cheddar cheese': ['grated sharp cheddar cheese']}"


In [23]:
second_map.to_csv("second_map.csv")

In [27]:
def extract_unique_ingredients_with_ids(df, ingredient_col="splitted_ingredients", id_col="ID", output_file="unique_ingredients.tsv"):
    ingredient_to_ids = {}

    for idx, row in df.iterrows():
        row_id = row[id_col]
        ingredients = row[ingredient_col]

        if not isinstance(ingredients, list):
            continue

        for sublist in ingredients:
            if not isinstance(sublist, list):
                continue  

            for ingredient in sublist:
                if ingredient not in ingredient_to_ids:
                    ingredient_to_ids[ingredient] = set()  
                ingredient_to_ids[ingredient].add(row_id)

    output_df = pd.DataFrame([
        (ingredient, ", ".join(map(str, sorted(ids))))  
        for ingredient, ids in ingredient_to_ids.items()
    ], columns=["Ingredient", "IDs"])

    output_df.to_csv(output_file, sep="\t", index=False)

In [28]:
extract_unique_ingredients_with_ids(ingredients)

In [26]:
ingredients.to_csv('ingredients.tsv', sep='\t', index=False)